# 广播与批量计算

学习目标：用广播完成按列中心化、按行缩放和批量计算，判断兼容形状与输出规模。

前置知识：数组形状与轴、逐元素运算、切片、newaxis、视图与副本。

运行环境：Python 3.12、NumPy 2.5。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例使用单元内构造的数据，后续单元沿用首次导入的 np。列均值和缩放系数直接给定。

标明“预期异常”的单元会直接显示原始报错；阅读异常类型与原因后，继续运行下一单元。

## 1 按列中心化

下面的 readings 有两次观测、三个传感器，形状为 (2, 3)。已给定三个传感器各自的列均值，每个读数减去对应列均值，称为按列中心化。

column_means 的形状为 (3,)，可直接从每一行减去。NumPy 按形状规则重复使用这些值，这种计算方式称为广播（broadcasting）。

In [1]:
import numpy as np

readings = np.array([[18.0, 30.0, 42.0], [22.0, 34.0, 38.0]])
column_means = np.array([20.0, 32.0, 40.0])
centered = readings - column_means

print(centered)  # 两行为 [-2. -2. 2.]、[2. 2. -2.]，每列减去对应均值。
print(centered.shape, centered.dtype)  # (2, 3) float64。
print(readings)  # 原读数保持不变；计算产生新的结果数组。

[[-2. -2.  2.]
 [ 2.  2. -2.]]
(2, 3) float64
[[18. 30. 42.]
 [22. 34. 38.]]


## 2 从尾轴判断形状

两个数组运算时，从最右侧的轴开始向左比较，缺少的前置轴按长度 1 处理。每对轴长度相等，或其中一个长度为 1，就能匹配；否则不能广播。相等的轴保留长度，长度 1 的轴按另一侧长度参与计算。

| 输入形状 | 匹配时的形状 | 结果形状 |
| --- | --- | --- |
| (2, 3) 与 (3,) | (2, 3) 与 (1, 3) | (2, 3) |
| (2, 3) 与 (2, 1) | (2, 3) 与 (2, 1) | (2, 3) |
| (2, 1) 与 (3,) | (2, 1) 与 (1, 3) | (2, 3) |

表中第三行会生成两个输入的全部组合。下面的官方图用四个基准 0、10、20、30 与三个调整量 1、2、3 展示同一规则：先找出一行与一列的对应值，再核对相加结果。

![NumPy 官方广播图：形状 (4, 1) 与 (3,) 的两个数组相加，得到四行三列的所有组合；浅色格表示概念上的扩展。](image/illustration/07-01-broadcast-grid.png)

引用原图：NumPy 文档 Broadcasting，Figure 4，NumPy Developers，BSD-3-Clause。原图未改绘；浅色格表示广播时的对应关系，不表示先复制两张完整矩阵，输入的 shape 也不因此改变。具体来源和许可见篇末。

下面的 bases 只取两个基准 10、20，因此得到六个组合、形状 (2, 3)，与原图的 (4, 3) 有所不同。用同一匹配规则核对代码结果。

In [2]:
bases = np.array([[10.0], [20.0]])
offsets = np.array([1.0, 2.0, 3.0])
combinations = bases + offsets

print(bases.shape, offsets.shape)  # (2, 1) (3,)，输入形状没有改变。
print(combinations)  # 两行为 [11. 12. 13.]、[21. 22. 23.]。
print(combinations.shape)  # (2, 3)，两个输入都有值被重复使用。

(2, 1) (3,)
[[11. 12. 13.]
 [21. 22. 23.]]
(2, 3)


## 3 标量计算

一个修正量作用于所有读数时，直接使用标量。标量没有轴，可以与任意形状的数组广播；输出保留数组形状。

下面给两次观测、三个传感器的温度统一增加 0.5 °C。

In [3]:
temperatures = np.array([[18.0, 20.0, 22.0], [19.0, 21.0, 23.0]])
adjusted = temperatures + 0.5

print(adjusted)  # 两行为 [18.5 20.5 22.5]、[19.5 21.5 23.5] °C。
print(adjusted.shape, adjusted.dtype)  # (2, 3) float64。

[[18.5 20.5 22.5]
 [19.5 21.5 23.5]]
(2, 3) float64


## 4 行列方向与 newaxis

### 4.1 每行使用不同系数

给两行分别乘以 2.0 和 0.5，需要让系数沿行轴变化、沿列轴重复。np.newaxis 增加长度为 1 的轴，将 (2,) 改为 (2, 1)，再与 (2, 3) 运算。

一维数组没有单独的行列方向。表达方向的是轴的位置和长度，不是变量名。

In [4]:
values = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
row_scales = np.array([2.0, 0.5])
scale_column = row_scales[:, np.newaxis]
scaled = values * scale_column

print(scale_column, scale_column.shape)  # 两行一列，shape 为 (2, 1)。
print(scaled)  # 首行乘 2 得 [2. 4. 6.]，末行乘 0.5 得 [2. 2.5 3.]。
print(scaled.shape, scaled.dtype)  # (2, 3) float64。

[[2. ]
 [0.5]] (2, 1)
[[2.  4.  6. ]
 [2.  2.5 3. ]]
(2, 3) float64


### 4.2 每列使用不同系数

若三个系数分别对应三列，可以直接使用形状 (3,)；也可以用 np.newaxis 明确写成 (1, 3)。两种形状都沿行轴重复。

下面仍使用上节的 values，但改成按列缩放。

In [5]:
column_scales = np.array([1.0, 10.0, 100.0])
scale_row = column_scales[np.newaxis, :]

print(scale_row.shape)  # (1, 3)，只有一行，包含三个列系数。
print(values * column_scales)  # 两行为 [1. 20. 300.]、[4. 50. 600.]。
print(values * scale_row)  # 与上一结果相同，仍为 (2, 3)。

(1, 3)
[[  1.  20. 300.]
 [  4.  50. 600.]]
[[  1.  20. 300.]
 [  4.  50. 600.]]


## 5 不兼容的形状

(2, 3) 与 (2,) 的尾轴长度分别为 3 和 2，既不相等也没有 1，因此不能广播。元素总数、变量名和“想按行计算”的意图都不能代替形状条件。

下面先观察错误，再用 newaxis 明确行系数方向。

In [6]:
values = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
row_scales = np.array([2.0, 0.5])

# 预期 ValueError：从尾轴对齐时长度 3 与 2 不同，且都不是 1，无法广播。
values * row_scales

ValueError: operands could not be broadcast together with shapes (2,3) (2,) 

In [7]:
print(values * row_scales[:, np.newaxis])
# 两行分别按各自系数计算，结果为 [[2. 4. 6.] [2. 2.5 3.]]。

[[2.  4.  6. ]
 [2.  2.5 3. ]]


## 6 批量维度

形状 (2, 3, 4) 可以表示两批数据、每批三次观测、每次四个通道。形状 (4,) 的通道修正量与尾轴匹配，同一套修正量用于每一批、每一次观测。

每批使用一个缩放系数时，将 (2,) 写成 (2, 1, 1)，在观测轴和通道轴上重复。先判断输出仍为 (2, 3, 4)，再计算。

In [8]:
batches = np.array([
    [[11.0, 12.0, 13.0, 14.0], [21.0, 22.0, 23.0, 24.0],
     [31.0, 32.0, 33.0, 34.0]],
    [[111.0, 112.0, 113.0, 114.0], [121.0, 122.0, 123.0, 124.0],
     [131.0, 132.0, 133.0, 134.0]],
])
channel_offsets = np.array([1.0, 2.0, 3.0, 4.0])
batch_scales = np.array([1.0, 2.0])

corrected = batches - channel_offsets
# batch_scales 变为 (2, 1, 1)，每批系数沿通道与时刻两个轴展开。
scaled = corrected * batch_scales[:, np.newaxis, np.newaxis]

print(corrected[:, 0, :])  # 两批的首次观测分别为四个 10、四个 110。
print(scaled[:, 0, :])  # 第二批乘 2，变为四个 220；第一批仍为四个 10。
print(scaled.shape, scaled.dtype)  # (2, 3, 4) float64。

[[ 10.  10.  10.  10.]
 [110. 110. 110. 110.]]
[[ 10.  10.  10.  10.]
 [220. 220. 220. 220.]]
(2, 3, 4) float64


## 7 广播与显式重复

为计算而广播，不需要先把小数组复制成大数组。np.tile() 则显式构造重复结果；本例的 (2, 1) 表示沿第一个轴重复两次、沿第二个轴重复一次。

只为逐元素计算时，直接广播即可。下面比较同一任务的两种写法；nbytes 表示数组元素占用的字节数，不包含数组对象的其他开销。

In [9]:
values = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
offsets = np.array([1.0, 2.0, 3.0])
repeated = np.tile(offsets, (2, 1))

print(repeated)  # 两行均为 [1. 2. 3.]，显式保存了重复值。
print(values - offsets)  # 直接广播：两行均分别为 [0. 0. 0.]、[3. 3. 3.]。
print(values - repeated)  # 数值相同，但提前构造了重复数组。
print(offsets.nbytes, repeated.nbytes)  # 24、48 字节，均使用 float64。
print(np.shares_memory(offsets, repeated))  # False，本例重复数组不共享元素内存。

[[1. 2. 3.]
 [1. 2. 3.]]
[[0. 0. 0.]
 [3. 3. 3.]]
[[0. 0. 0.]
 [3. 3. 3.]]
24 48
False


## 8 输出与中间数组的规模

广播节省了重复输入的存储，但计算结果仍可能很大。(3, 1) 与 (3,) 运算会产生 (3, 3)，不会得到三个对应位置的结果。

下面对比“逐对作差”和“所有组合的差”。如果继续对差值数组平方，差值和平方结果都是需要考虑的数组；nbytes 不是进程峰值内存。

In [10]:
observed = np.array([10.0, 20.0, 30.0])
reference = np.array([1.0, 2.0, 3.0])
paired = observed - reference
all_pairs = observed[:, np.newaxis] - reference
squared = all_pairs * all_pairs

print(paired, paired.shape)  # [9. 18. 27.] (3,)，只计算三个对应位置。
print(all_pairs)  # 三行为 [9. 8. 7.]、[19. 18. 17.]、[29. 28. 27.]。
print(all_pairs.shape, all_pairs.nbytes)  # (3, 3) 72。
print(squared.shape, squared.nbytes)  # (3, 3) 72，另有同样大小的平方结果。

# 只估算，不创建大数组：10 000 个值两两作差会得到 1 亿个 float64。
count = 10_000
estimated_bytes = count * count * observed.itemsize
print(estimated_bytes)  # 800000000 字节，仅计一个结果的元素数据。

[ 9. 18. 27.] (3,)
[[ 9.  8.  7.]
 [19. 18. 17.]
 [29. 28. 27.]]
(3, 3) 72
(3, 3) 72
800000000


## 9 综合应用：中心化后按行缩放

下面有三次观测、两个特征，列均值和行系数均已给定。先让每列减去自己的均值，再让每行乘以自己的系数，保留形状 (3, 2)。

两个参数的形状不同：列均值为 (2,)，行系数需要 (3, 1)。这是两次分别匹配轴的逐元素计算。

In [11]:
values = np.array([[8.0, 18.0], [10.0, 20.0], [12.0, 22.0]])
column_means = np.array([10.0, 20.0])
row_scales = np.array([1.0, 2.0, 0.5])

centered = values - column_means
scaled = centered * row_scales[:, np.newaxis]

print(centered)  # 三行为 [-2. -2.]、[0. 0.]、[2. 2.]。
print(scaled)  # 三行为 [-2. -2.]、[0. 0.]、[1. 1.]。
print(scaled.shape, scaled.dtype)  # (3, 2) float64。
print(values)  # 原始三行数据保持不变。

[[-2. -2.]
 [ 0.  0.]
 [ 2.  2.]]
[[-2. -2.]
 [ 0.  0.]
 [ 1.  1.]]
(3, 2) float64
[[ 8. 18.]
 [10. 20.]
 [12. 22.]]


## 10 选学：显式广播视图

### 10.1 broadcast_to

np.broadcast_to() 把输入呈现为指定的兼容形状，返回只读视图。多个位置可能引用同一份元素内存，不能把显示出的重复值当作各自独立的数据。

需要逐位置修改时，先 copy()。下面不更改视图的写入标志。

In [12]:
offsets = np.array([1.0, 2.0, 3.0])
expanded = np.broadcast_to(offsets, (2, 3))

print(expanded)  # 两行均为 [1. 2. 3.]。
print(expanded.shape, expanded.dtype)  # (2, 3) float64。
print(expanded.flags.writeable)  # False，只读视图。
print(np.shares_memory(offsets, expanded))  # True，共享原数组元素内存。

[[1. 2. 3.]
 [1. 2. 3.]]
(2, 3) float64
False
True


In [13]:
# 预期 ValueError：broadcast_to 返回的共享视图只读，不能直接赋值。
expanded[0, 0] = 99.0

ValueError: assignment destination is read-only

In [14]:
editable = expanded.copy()
editable[0, 0] = 99.0
print(editable)  # 仅第一行第一个值改为 99.0，第二行第一个仍为 1.0。
print(offsets)  # 原数组仍为 [1. 2. 3.]。

[[99.  2.  3.]
 [ 1.  2.  3.]]
[1. 2. 3.]


### 10.2 broadcast_arrays

np.broadcast_arrays() 接收多个输入，返回广播到共同形状的数组视图，可用于查看参与计算的值如何对应。

NumPy 2.5 文档说明，这些视图可能有多个位置指向同一内存，直接写入属于弃用行为，会有警告，并计划在后续版本禁止。需要写入时先复制；不要依赖它当前允许写入，也不要把它与 broadcast_to() 的只读保证混为一谈。

In [15]:
row = np.array([[1.0, 2.0, 3.0]])
column = np.array([[10.0], [20.0]])
row_view, column_view = np.broadcast_arrays(row, column)

print(row_view)  # 两行均为 [1. 2. 3.]。
print(column_view)  # 第一行全为 10.0，第二行全为 20.0。
print(row_view.shape, column_view.shape)  # 两者均为 (2, 3)。
print(np.shares_memory(row, row_view))  # True。
print(np.shares_memory(column, column_view))  # True。

editable = column_view.copy()
editable[0, 1] = 99.0
print(editable)  # 第一行为 [10. 99. 10.]，只改一个独立位置。
print(column)  # 原输入仍是两行一列，分别为 10.0、20.0。

[[1. 2. 3.]
 [1. 2. 3.]]
[[10. 10. 10.]
 [20. 20. 20.]]
(2, 3) (2, 3)
True
True
[[10. 99. 10.]
 [20. 20. 20.]]
[[10.]
 [20.]]


## 本章小结

（1）广播从尾轴向左匹配，每对轴长度相等或其中一个为 1；缺失的前置轴按 1 处理。

（2）按列计算可用尾轴匹配的一维参数，按行计算常用 newaxis 把参数变成一列。批量轴同样按形状匹配。

（3）广播无需先重复输入，但输出和中间数组仍占用存储。运行前应能判断 shape 与元素数。

（4）broadcast_to() 返回只读视图，broadcast_arrays() 的输出也不应直接写入；需要独立修改时先复制。

## 练习

（1）用已给定的列均值中心化下表，再按给定行系数缩放。打印每一步的结果、shape 和 dtype。

In [16]:
values = np.array([[8.0, 16.0, 30.0], [12.0, 24.0, 34.0]])
column_means = np.array([10.0, 20.0, 32.0])
row_scales = np.array([0.5, 2.0])

# 在此完成中心化，再把行系数改为正确方向进行缩放。
# 检查：中心化后首行为 [-2. -4. -2.]，末行为 [2. 4. 2.]。
# 缩放后首行为 [-1. -2. -1.]，末行为 [4. 8. 4.]，shape 仍为 (2, 3)。

（2）先预测下面三个结果的 shape，再运行核对。解释 newaxis 放在不同位置时，哪个轴会重复使用数据。

In [17]:
first = np.array([1.0, 2.0, 3.0])
second = np.array([10.0, 20.0, 30.0])

# 先记录形状预测，再检查结果中的元素对应关系。
print((first + second).shape)
print((first[:, np.newaxis] + second).shape)
print((first[np.newaxis, :] + second[:, np.newaxis]).shape)

# 在此用注释解释：哪些表达式计算对应位置，哪些计算所有组合？

(3,)
(3, 3)
(3, 3)


（3）有四条观测，每条有两个特征，要让每条观测乘以一个系数。不能构造完整的重复系数表。从直接使用一维系数、newaxis 增轴、tile 重复三种方案中选择，说明理由，再计算。

In [18]:
values = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0], [7.0, 8.0]])
scales = np.array([1.0, 2.0, 3.0, 4.0])

# 在此说明方案选择及另外两种方案不符合本题的原因，然后计算。
# 检查：输出 shape 为 (4, 2)，四行依次为 [1. 2.]、[6. 8.]、[15. 18.]、[28. 32.]。
# 继续判断：若误把两个长度为 10 000 的一维数组写成列与行后相加，会产生多少元素？
# 只计算规模，不分配大数组。

（4）下面两批数据使用同一套通道修正量，先减去修正量，再分别乘以批次系数。解释参数应在哪些位置增加长度为 1 的轴。

In [19]:
batches = np.array([
    [[2.0, 4.0, 6.0], [3.0, 5.0, 7.0]],
    [[12.0, 14.0, 16.0], [13.0, 15.0, 17.0]],
])
channel_offsets = np.array([1.0, 2.0, 3.0])
batch_scales = np.array([1.0, 0.5])

# 轴依次表示批次、观测、通道；在此完成修正和批次缩放。
# 检查：输出仍为 (2, 2, 3)，第一批首次观测为 [1. 2. 3.]。
# 第二批首次观测为 [5.5 6. 6.5]，输入数组保持不变。

### 重点练习提示

对应第（3）题。先独立完成，再按需要查看提示。

（1）系数对应行，先写出它在两个轴上的目标含义。

（2）将 (4,) 改为 (4, 1)，让长度为 1 的特征轴广播到两列；只估算大数组的元素数量。

### 重点练习参考解析

对应第（3）题。

选择 newaxis 增轴，使 scales[:, np.newaxis] 的形状为 (4, 1)，再与 (4, 2) 的 values 相乘。四行结果为 [1, 2]、[6, 8]、[15, 18]、[28, 32]，形状仍是 (4, 2)，dtype 为 float64。

直接用 (4,) 会把 4 与最后一轴的 2 对齐，形状不兼容；tile 会构造重复系数表，违反约束。两个长度为 10000 的向量若变成列与行再相加，输出有 100000000 个元素；若输出为 float64，仅元素数据就需 800000000 字节。无需实际分配来判断这个成本。

## 参考与引用来源

第 2 节引用 NumPy 官方广播原图，未改绘；该图不作为本课程的实际运行截图。图片版权及 BSD-3-Clause 完整许可条件、免责声明保存在[随图许可](image/illustration/07-01-broadcast-grid.LICENSE.txt)。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| NumPy 官方文档（NumPy 2.5） | [Broadcasting](https://numpy.org/doc/2.5/user/basics.broadcasting.html) 的标量示例、General broadcasting rules、Broadcastable arrays（含 Figure 4）；[Figure 4 原图](https://numpy.org/doc/2.5/_images/broadcasting_4.png)；A practical example: vector quantization 中的大型中间数组讨论；[newaxis](https://numpy.org/doc/2.5/reference/constants.html#numpy.newaxis) 的增轴及行列组合示例；[tile](https://numpy.org/doc/2.5/reference/generated/numpy.tile.html) 的 reps 与优先使用广播的说明；[ndarray.nbytes](https://numpy.org/doc/2.5/reference/generated/numpy.ndarray.nbytes.html) 的元素字节数与 Notes；[shares_memory](https://numpy.org/doc/2.5/reference/generated/numpy.shares_memory.html) 的共享元素判断；[broadcast_to](https://numpy.org/doc/2.5/reference/generated/numpy.broadcast_to.html) 的只读视图 Returns；[broadcast_arrays](https://numpy.org/doc/2.5/reference/generated/numpy.broadcast_arrays.html) 的视图、重叠位置与写入弃用说明；[ndarray.copy](https://numpy.org/doc/2.5/reference/generated/numpy.ndarray.copy.html) 的独立数值副本。 |
| GitHub（NumPy 官方仓库） | [v2.5.0 LICENSE.txt](https://github.com/numpy/numpy/blob/v2.5.0/LICENSE.txt)：NumPy Developers 版权声明、BSD-3-Clause 许可条件及免责声明，适用于本章引用的广播原图。 |